# Integrative Industry Synthesis (Revised)
## AI-Assisted Chronic Care Triage for Diabetes Follow-Up

This notebook walks through the **revised** integrated healthcare AI workflow that combines:
1. data analysis and preprocessing on the **UCI Diabetes 130-US Hospitals (1999-2008)** dataset,
2. supervised machine learning (logistic regression + random forest, selected on held-out ROC-AUC),
3. an LLM-driven generative layer for clinical case summaries and patient outreach messages, and
4. an LLM-driven agentic routing layer that selects a follow-up tool and produces a JSON justification.

The earlier version of this artifact used synthetic data and rule-based string templates for the generative and agentic layers. Both have been replaced. See `Reflective_Synthesis_Paper.md` and the README for the full revision note.

**Before running:** copy `.env.example` to `.env` and set at least one of `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, or `LOCAL_LLM_GGUF`.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.append(str(ROOT / 'src'))

from healthcare_triage_ai import (
    load_uci_diabetes_dataset,
    preprocess_dataset,
    train_models,
    save_visuals,
    shortlist_for_llm,
    run_llm_layers,
    export_outputs,
    LLMClient,
    AGENT_TOOLS,
    LLM_SHORTLIST_SIZE,
)

## Step 1 — Load and preprocess the real dataset

The UCI dataset (~99,000 inpatient encounters across 130 U.S. hospitals) is fetched on first use and cached locally at `data/diabetic_data.csv`.

Target definition: `needs_intervention = 1` when the patient was readmitted in fewer than 30 days, and `0` otherwise.

In [ ]:
raw = load_uci_diabetes_dataset()
df = preprocess_dataset(raw)
print(f'rows: {len(df):,}')
print(f'positive rate (30-day readmit): {df["needs_intervention"].mean():.3f}')
df.head()

## Step 2 — Train two classifiers and select the better one

Both models are evaluated on the same stratified 80/20 split, with class-weight balancing because the positive class is roughly 11% of the cohort. The model with the higher held-out ROC-AUC is used to score everyone.

In [ ]:
scored_df, metrics, model = train_models(df)
save_visuals(scored_df, model, metrics)
print(json.dumps({k: v for k, v in metrics.items() if k != 'models'}, indent=2))
print('\nPer-model metrics:')
for name, info in metrics['models'].items():
    print(f'  - {name}: {info["metrics"]}')
scored_df['risk_band'].value_counts()

## Step 3 — LLM-driven generative + agentic layers on the triage shortlist

The classifier scores all ~99k patients. The LLM layers run only on a shortlist (default 20 patients, configurable via `LLM_SHORTLIST_SIZE`). For each shortlisted patient the system produces:

- a **case summary** for the care team (LLM)
- a **patient outreach message** (LLM)
- an **agent decision** (`{action, justification, follow_up_hours}`) selected from a fixed tool set (LLM, JSON-mode)

Available agent tools:

In [ ]:
for tool in AGENT_TOOLS:
    print(f"- {tool['name']}: {tool['description']}\n")

In [ ]:
llm = LLMClient()
print(f'LLM backend: {llm.backend} ({llm.model_name})')
shortlist = shortlist_for_llm(scored_df, LLM_SHORTLIST_SIZE)
llm_df = run_llm_layers(llm, shortlist)
llm_df['agent_action'].value_counts()

## Step 4 — Inspect a few routed cases

In [ ]:
preview = llm_df.sort_values('risk_probability', ascending=False).head(5)
for _, row in preview.iterrows():
    print('=' * 80)
    print(f"Patient {row['patient_id']} | risk={row['risk_probability']:.1%} ({row['risk_band']})")
    print(f"Agent action: {row['agent_action']} (follow up within {row['follow_up_hours']} h)")
    print(f"Justification: {row['agent_justification']}")
    print(f"Case summary:  {row['case_summary']}")
    print(f"Patient msg:   {row['patient_message']}")

## Step 5 — Export the same artifacts as the script

In [ ]:
llm_meta = {
    'backend': llm.backend,
    'model': llm.model_name,
    'shortlist_size': int(len(shortlist)),
    'tools': [tool['name'] for tool in AGENT_TOOLS],
}
export_outputs(scored_df, llm_df, metrics, llm_meta)

## Responsible AI Notes
- Real but **historical** data (1999-2008 U.S. inpatient). Generalization to current outpatient populations is not assumed.
- LLM prompts forbid diagnosis, prescription, and invented indicators.
- The agent's action space is closed and validated; invalid output is conservatively escalated to physician review.
- High-risk and ambiguous cases are routed to **human review**.
- Real deployment would require subgroup fairness analysis, governance review, an explicit PHI policy for any third-party LLM call (or use of the local `llama-cpp` backend), and ongoing monitoring.